# 📗 양자화와 추론 최적화: 정밀도와 양자화로 모델을 가볍게 실행하기

교안 01에서 BF16으로 실행한 Qwen3.5-4B는 가중치만 약 7.8GiB입니다. 이 교안은 모델 메모리를 계산하고, 가중치를 적은 비트의 정수로 담는 양자화(quantization)를 직접 해 오차와 답의 변화를 확인한 뒤, 같은 모델의 Ollama 양자화 태그를 속도와 답으로 비교합니다. 1절에서 핵심 코드를 먼저 실행하고, 2절부터 그 코드를 한 부분씩 풀어 봅니다.

<img src="images/교안/30_lesson02_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 정밀도와 양자화로 모델을 가볍게 실행하기'입니다. 띠: Qwen3.5-4B의 크기 · 속도 · 답 비교. 1 메모리 계산(파라미터 수 × 바이트): 높이가 줄어드는 막대 네 개. 2 정밀도(FP32 · BF16 · FP16): 긴 비트 띠 하나와 절반 길이 띠 두 개. 3 INT8 양자화(스케일 · 묶음): 실수 점들이 정수 눈금으로 옮겨짐. 4 양자화 형식 고르기(AWQ · GPTQ · GGUF): 개발자가 크기가 다른 파일 가운데 작은 파일을 가리킴.">

**학습 목표** (그림의 번호와 같습니다)

1. **파라미터 수와 정밀도로 모델 메모리를 계산하고, 생성 여유분까지 더해 GPU에 들어가는지 판정합니다.**
2. **FP32·BF16·FP16의 비트 구성과 범위·자릿수 차이를 설명하고, 같은 값을 세 정밀도로 바꿔 확인합니다.**
3. **대칭 양자화를 직접 계산하고, 스케일을 묶음마다 두면 이상치가 섞인 가중치의 오차와 4비트 모델의 답이 어떻게 달라지는지 확인합니다.**
4. **AWQ·GPTQ·GGUF의 차이를 설명하고, Ollama 양자화 태그를 크기·토큰/초·답으로 비교해 형식을 고릅니다.**

아래 셀은 이 교안에서 쓸 모델 파일을 미리 내려받습니다. 처음 한 번은 몇 분 걸리므로 먼저 실행해 두고 아래 설명을 읽습니다. Ollama 서버가 켜져 있어야 합니다(설치 가이드 2절). BF16 원본은 Transformers로 불러오므로 Ollama의 `bf16` 태그는 받지 않습니다.

In [ ]:
import asyncio

from huggingface_hub import snapshot_download
import ollama

# 실습에서 쓸 모델을 미리 내려받습니다. 처음 한 번만 오래 걸리고, 다음부터는 이미 받은 파일을 확인만 합니다
# Transformers용 원본 모델(BF16, 약 9GB): Hugging Face 캐시 폴더(기본 ~/.cache/huggingface/hub)에 저장됩니다
# snapshot_download는 일반 함수라서 asyncio.to_thread로 감싸야 다른 다운로드와 함께 기다릴 수 있습니다
tasks = [asyncio.to_thread(snapshot_download, "Qwen/Qwen3.5-4B")]

# Ollama용 모델(GGUF 파일): pull 요청을 받은 Ollama 서버가 내려받아 서버의 모델 폴더(기본 ~/.ollama/models)에 보관합니다
# 태그 이름은 ollama.com/search 에서 모델을 검색하고 모델 페이지의 Tags 탭에서 고릅니다
ollama_client = ollama.AsyncClient()
ollama_tags = ["qwen3.5:4b-q8_0", "qwen3.5:4b-q4_K_M"]
for tag in ollama_tags:
    tasks.append(ollama_client.pull(tag))

# 두 다운로드는 받는 곳과 저장 위치가 달라 서로 기다릴 필요가 없으므로 동시에 받고, 결과는 tasks 순서대로 받습니다
results = await asyncio.gather(*tasks)
model_dir = results[0]
print("Hugging Face 모델:", model_dir)
print("Ollama 모델:", ollama_tags)

## 1. 핵심 코드 먼저 보기

이 교안의 핵심 코드 두 개를 실제 모델에 먼저 실행해 결과를 봅니다. 왜 그런 결과가 나오는지는 2절부터 하나씩 설명합니다.

| 핵심 코드 | 하는 일 | 설명하는 절 |
|---|---|---|
| 1-2 `model_memory_gib` | 정밀도마다 모델 메모리 계산 | 2절 정밀도와 모델 메모리 |
| 1-3 `measure_tag` | 같은 모델의 Ollama 8비트·4비트 태그로 토큰/초와 답 비교 | 6절 양자화 형식, 7절 결과 읽기 |

3~5절에서는 8비트·4비트가 무엇인지, 가중치를 정수로 바꾸는 양자화를 직접 계산해 봅니다.

### 1-1. 준비 셀

계산 장치는 지난 단원과 같은 코드입니다. 모델은 교안 01과 같은 코드로 BF16 그대로 불러옵니다.

- **앞 커널 종료:** 교안 01(또는 앞 노트북)의 커널을 종료(Shut Down)한 뒤 실행합니다. 켜 둔 채 실행하면 같은 모델이 GPU에 두 벌 올라갑니다.
- **Ollama 모델 내리기:** 모델을 불러오기 전에 Ollama가 GPU에 올려 둔 모델을 `keep_alive=0`으로 내려, Transformers 모델을 올릴 GPU 메모리를 비웁니다. 목록이 빈 것(`[]`)을 확인합니다.
- **경고:** `HF_TOKEN`을 설정하라는 경고는 로그인하지 않고 공개 모델을 받는다는 안내입니다. Gated DeltaNet 층(교안 01 3-5)의 빠른 계산 패키지(flash-linear-attention 등)가 없다는 경고가 나와도 결과는 같고 속도만 느립니다. 둘 다 그대로 진행합니다.

In [ ]:
import pandas as pd

import torch

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블에서 긴 답이 잘리지 않게 합니다
pd.set_option("display.max_colwidth", None)

In [ ]:
# 앞에서 Ollama가 GPU에 올려 둔 모델이 있으면 내립니다. keep_alive=0은 "지금 바로 GPU에서 내리기"라는 뜻입니다
for loaded in ollama.ps().models:
    ollama.generate(model=loaded.model, keep_alive=0)
print("Ollama가 GPU에 올린 모델:", ollama.ps().models)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

# 토크나이저를 불러옵니다. 파일은 맨 위 미리 내려받기 셀이 받아 두었으므로 캐시에서 읽습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen3.5-4B는 이미지도 받는 모델이지만, AutoModelForCausalLM은 글 생성 부분만 불러옵니다
# 모델 파일에 저장된 정밀도(BF16) 그대로 계산 장치에 올립니다(GPU 메모리 약 8GB). 불러온 모델은 이미 평가 모드입니다
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)
print("불러온 모델:", MODEL_NAME, "| 정밀도:", model.dtype, "| 장치:", model.device)

### 1-2. 모델 메모리 계산

불러온 모델의 파라미터 수에 정밀도마다 파라미터 하나의 바이트를 곱해 메모리를 계산합니다. 생성할 때 더 드는 메모리는 20%로 어림합니다.

In [ ]:
def model_memory_gib(n_params, bytes_per_param):
    """파라미터 수 * 파라미터 하나의 바이트를 GiB로 바꿔 돌려줍니다."""
    return n_params * bytes_per_param / 1024 ** 3


# 정밀도마다 파라미터 하나의 바이트
precision_bytes = {"FP32": 4, "BF16": 2, "INT8": 1, "INT4": 0.5}

# 불러온 모델의 파라미터 수를 셉니다
total_params = sum(p.numel() for p in model.parameters())
print("파라미터 수:", total_params)

# 정밀도마다 가중치 메모리와, 20%를 더한 생성할 때 메모리를 계산합니다
memory_rows = []
for precision, n_bytes in precision_bytes.items():
    weight_gib = model_memory_gib(total_params, n_bytes)
    memory_rows.append({"정밀도": precision, "가중치(GiB)": weight_gib, "생성 시(GiB)": weight_gib * 1.2})
display(pd.DataFrame(memory_rows).round(2))

- 정밀도를 절반으로 줄일 때마다 메모리도 절반이 됩니다. 같은 GPU에 더 큰 모델을 올리거나 더 빨리 생성하려면 비트를 줄입니다.

### 1-3. Ollama 양자화 태그의 속도와 답

맨 위 셀에서 받은 두 태그는 같은 Qwen3.5-4B를 8비트(`q8_0`)와 4비트(`q4_K_M`)로 담은 GGUF 파일입니다. 같은 질문으로 1초에 생성하는 토큰 수(토큰/초)와 답을 비교합니다.

In [ ]:
import statistics


def measure_tag(tag, messages):
    """tag로 한 번 미리 생성한 뒤 3번 생성해 토큰/초 중앙값과 마지막 답을 돌려주고, 모델을 GPU에서 내립니다."""
    # 같은 답이 나오도록 temperature 0, seed 0으로 두고 최대 150토큰까지 생성합니다
    options = {"temperature": 0, "seed": 0, "num_predict": 150}

    # 첫 호출은 모델을 GPU에 올리고 준비하는 일이 겹쳐 생성 속도가 흔들리므로 결과를 버립니다
    ollama.chat(model=tag, messages=messages, think=False, options=options)

    # 3번 생성해 토큰/초를 모읍니다. eval_duration은 나노초 단위라 1e9를 곱해 초 단위로 맞춥니다
    speeds = []
    for _ in range(3):
        response = ollama.chat(model=tag, messages=messages, think=False, options=options)
        speeds.append(response.eval_count / response.eval_duration * 1e9)

    # 다음 태그 측정에 이 모델이 섞이지 않게 GPU에서 내립니다
    ollama.generate(model=tag, keep_alive=0)
    return statistics.median(speeds), response.message.content


# 교안 01과 같은 헬프데스크 system 메시지와 질문 하나를 준비합니다
helpdesk_system = "당신은 사내 IT 헬프데스크 안내원입니다. 직원의 질문에 한국어로 짧고 단계별로 답합니다."
helpdesk_question = "VPN이 자꾸 끊길 때 확인할 점을 알려 주세요."
helpdesk_messages = [{"role": "system", "content": helpdesk_system}, {"role": "user", "content": helpdesk_question}]

# 8비트와 4비트 태그에 같은 질문을 넣어 토큰/초 중앙값과 답을 비교합니다
tags = ["qwen3.5:4b-q8_0", "qwen3.5:4b-q4_K_M"]
speed_rows = []
for tag in tags:
    speed_median, answer = measure_tag(tag, helpdesk_messages)
    speed_rows.append({"태그": tag, "토큰/초 중앙값": speed_median, "답": answer})
display(pd.DataFrame(speed_rows).round(1))

- 보통 4비트 태그가 8비트 태그보다 빠릅니다. 두 태그의 속도 차이와 답이 같은지는 출력에서 확인합니다.

## 2. 정밀도와 모델 메모리

1-2의 `precision_bytes`는 정밀도마다 파라미터 하나의 바이트입니다. 이 절은 그 정밀도가 무엇인지, 그리고 1-2의 계산이 실제 메모리와 맞는지 확인합니다.

### 2-1. 부동소수의 구성

교안 01에서 정밀도를 파라미터 하나의 바이트로 봤습니다. 같은 16비트인 BF16과 FP16이 어떻게 다른지 비트 구성으로 봅니다. **부동소수**(floating point)는 실수 하나를 세 부분으로 나눠 담는 형식입니다.

- **부호(sign):** 양수인지 음수인지(1비트)
- **지수(exponent):** 2의 몇 제곱인지. 지수 비트가 많을수록 아주 큰 수와 아주 작은 수까지 담습니다(범위).
- **가수(mantissa):** 유효 숫자. 가수 비트가 많을수록 촘촘하게 담습니다(자릿수).

<img src="images/교안/32_float_formats.png" width="1050" alt="제목은 '부동소수 형식: 부호 · 지수 · 가수'입니다. FP32: 부호 1, 지수 8, 가수 23, 범위가 넓고 자릿수가 많음. BF16: 부호 1, 지수 8, 가수 7, 범위는 FP32와 같고 자릿수가 적음. FP16: 부호 1, 지수 5, 가수 10, 자릿수는 더 많고 최댓값은 65504. 결론: 지수 비트는 범위를, 가수 비트는 자릿수를 정합니다.">

`torch.finfo(dtype)`는 그 정밀도가 담는 최댓값(`max`), 가장 작은 정규 양수(`tiny`), 1 근처의 눈금 간격(`eps`, 자릿수)을 알려 줍니다.

In [ ]:
# 세 정밀도의 비트 수, 최댓값, 가장 작은 정규 값, 1 근처의 눈금 간격을 테이블로 봅니다
finfo_rows = []
for precision, dtype in [("FP32", torch.float32), ("BF16", torch.bfloat16), ("FP16", torch.float16)]:
    fp_info = torch.finfo(dtype)
    finfo_rows.append({"정밀도": precision, "비트": fp_info.bits, "max": fp_info.max, "tiny": fp_info.tiny, "eps": fp_info.eps})
display(pd.DataFrame(finfo_rows))

**테이블 읽기**
- **BF16:** 지수 비트가 FP32와 같아 `max`와 `tiny`가 FP32와 거의 같습니다. 대신 `eps`가 0.0078로 커서 유효 숫자가 약 2~3자리뿐입니다.
- **FP16:** `eps`가 0.00098로 BF16보다 자릿수가 조금 더 많지만, `max`가 65504라 범위가 좁습니다.

### 2-2. 같은 값을 세 정밀도로

큰 값, 자릿수가 많은 값, 1에 아주 가까운 값을 BF16과 FP16으로 바꿔 차이를 확인합니다.

In [ ]:
# 큰 값, 자릿수가 많은 값, 1에 아주 가까운 값을 FP32 텐서로 만듭니다
values = torch.tensor([70000.0, 3.14159265, 1.001])

# BF16과 FP16으로 바꾼 값을 원래 값과 나란히 놓습니다. 테이블에 넣으려고 .tolist()로 파이썬 숫자로 꺼냅니다
precision_table = pd.DataFrame({
    "FP32(원래 값)": values.tolist(),
    "BF16": values.to(torch.bfloat16).tolist(),
    "FP16": values.to(torch.float16).tolist(),
})
display(precision_table)

**테이블 읽기**
- **70000:** FP16은 최댓값 65504를 넘어 `inf`(무한대)가 됩니다. BF16은 범위가 넓어 담기는 하지만 자릿수가 모자라 70144가 됩니다.
- **3.14159265:** 두 정밀도 모두 3.140625가 됩니다.
- **1.001:** BF16은 1 근처 눈금 간격(`eps` 0.0078)보다 작은 차이를 담지 못해 1.0이 되고, FP16은 1.0009765625로 원래 값에 더 가깝습니다.

`inf`가 한 번 생기면 학습이 깨지므로, 최근 LLM은 범위가 넓은 BF16으로 학습하고 배포합니다. 신경망은 유효 숫자 2~3자리 수준의 오차를 견딥니다.

### 2-3. 파라미터 수 × 바이트

1-2의 `model_memory_gib`는 **파라미터 수 × 파라미터 하나의 바이트**를 GiB로 바꿉니다. 파라미터는 대부분 층의 행렬 값인 **가중치**라서 이 값을 가중치 메모리라고 부릅니다.

- **GPU 사양의 GB:** GiB로 읽습니다(교안 01 맨 위 단위 테이블).

어림이 실제와 맞는지 두 가지로 확인합니다. 하나는 모델 저장소의 파일 크기이고, 하나는 GPU에 올린 모델의 메모리입니다.

- **`HfApi().model_info(MODEL_NAME, files_metadata=True)`:** 저장소의 파일마다 크기(`size`, 바이트)를 함께 돌려줍니다. Hub 화면에서는 Files and versions 탭에 같은 크기가 나옵니다.
- **`model.get_memory_footprint()`:** 불러온 모델의 파라미터와 버퍼가 메모리에서 차지하는 바이트 수입니다.

In [ ]:
from huggingface_hub import HfApi

# 모델 저장소의 파일 목록을 크기와 함께 받아 와, 가중치 파일(.safetensors)의 크기를 모두 더합니다
repo_info = HfApi().model_info(MODEL_NAME, files_metadata=True)
safetensors_bytes = 0
for sibling in repo_info.siblings:
    if sibling.rfilename.endswith(".safetensors"):
        safetensors_bytes = safetensors_bytes + sibling.size

# 저장소 파일 크기, 불러온 모델을 BF16(2바이트)으로 어림한 크기, 실제로 메모리에 올린 크기를 GiB로 비교합니다
print("safetensors 파일 합(GiB):", round(safetensors_bytes / 1024 ** 3, 2))
print("불러온 파라미터 수 * 2바이트(GiB):", round(total_params * 2 / 1024 ** 3, 2))
print("불러온 메모리(GiB):", round(model.get_memory_footprint() / 1024 ** 3, 2))

- **메모리:** 불러온 메모리는 파라미터 수 × 2바이트 어림과 거의 같습니다. 가중치가 BF16으로 올라가 있다는 뜻입니다.
- **파일 합이 더 큰 이유:** 파일 합에는 비전 인코더와 mtp(다중 토큰 예측 보조층)가 들어 있어 불러온 메모리보다 큽니다(교안 01 5-1).

### 2-4. 생성할 때 더 드는 메모리와 GPU 판정

생성할 때는 임시 계산값과 KV 캐시(교안 01 5절)가 메모리를 더 씁니다. 1-2에서는 Hugging Face Accelerate 문서의 경험칙대로 가중치 메모리에 약 20%를 더했습니다. 입력이 길거나 여러 요청을 한 번에 처리하면 더 듭니다.

거꾸로 GPU 메모리가 정해져 있을 때 올릴 수 있는 모델 크기도 계산할 수 있습니다.

In [ ]:
# GPU 메모리에 20% 여유까지 넣었을 때 올릴 수 있는 최대 파라미터 수를 10억 개 단위로 계산합니다
fit_rows = []
for gpu_gib in [16, 24, 32, 80]:
    row = {"GPU 메모리(GB)": gpu_gib}
    for precision in ["BF16", "INT8", "INT4"]:
        max_params = gpu_gib * 1024 ** 3 / (precision_bytes[precision] * 1.2)
        row[f"{precision} 최대(10억 개)"] = max_params / 1e9
    fit_rows.append(row)
display(pd.DataFrame(fit_rows).round(1))

- **24GB GPU:** BF16으로 약 10.7B, INT8로 약 21.5B, INT4로 약 42.9B까지 올라갑니다. **32GB GPU**는 BF16으로 약 14.3B, INT8로 약 28.6B, INT4로 약 57.3B까지입니다.
- **실습 모델:** 글 생성 부분이 BF16 약 7.83GiB라 20%를 더해도 약 9.4GiB입니다. 16GB GPU에도 BF16 그대로 들어가고, 계산으로는 24GB GPU에 Ollama 4비트 모델과 함께 올릴 여유도 있습니다.
- 같은 GPU에 더 큰 모델을 올리려면 정밀도를 낮춰야 합니다. 그 방법이 양자화입니다.

## 3. 스케일과 묶음

16비트보다 더 줄이려면 실수 대신 정수로 저장합니다. 가중치 몇 개로 직접 계산해 보고, 3-2에서 함수로 만듭니다.

### 3-1. 대칭 INT8 양자화

- **양자화(quantization):** 가중치를 적은 비트의 정수와 배율로 바꿔 저장하는 것입니다.
- **스케일(scale):** 정수를 실수로 되돌리는 배율입니다. 대칭 양자화는 `scale = 최대 절댓값 / 127`로 정해, 절댓값이 가장 큰 값이 127에 오게 합니다.
- **대칭 양자화(symmetric quantization):** 0을 정수 0에 두고 양수와 음수를 같은 간격으로 담는 방식입니다. INT8은 -128~127을 담을 수 있지만 0을 가운데 두고 양쪽을 맞추려고 -127~127만 씁니다.
- **복원(역양자화, dequantization):** 저장한 정수에 스케일을 곱해 실수로 되돌리는 것입니다. 반올림한 만큼 원래 값과 차이가 나는데, 이 차이가 양자화 오차입니다.

<img src="images/교안/33_symmetric_int8.png" width="1050" alt="제목은 '대칭 INT8 양자화: 나누고, 반올림하고, 다시 곱하기'입니다. 실수 눈금 -0.51, 0, 0.51이 정수 눈금 -127, 0, 127로 옮겨짐. 스케일 = 최대 절댓값 ÷ 127 = 0.004016. 1 양자화: 0.10 ÷ 0.004016 = 24.9. 2 반올림해서 25: 정수 25 (1바이트). 3 복원: 25 × 0.004016 = 0.1004. 결론: 복원값과 원래 값의 차이가 양자화 오차입니다.">

In [ ]:
# 층 가중치 6개를 예로 듭니다. 스케일은 절댓값이 가장 큰 값이 127에 오도록 정합니다
weights = torch.tensor([0.10, -0.42, 0.03, 0.51, -0.28, 0.00])
scale = weights.abs().max() / 127
print("스케일:", scale.item())

In [ ]:
# 양자화: 스케일로 나누고 반올림해 8비트 정수로 저장합니다. 복원: 정수에 스케일을 다시 곱합니다
q = torch.round(weights / scale).to(torch.int8)
restored = q.float() * scale

# 원래 값, 정수, 복원값, 오차(원래 값 - 복원값)를 한 테이블로 봅니다
int8_example = pd.DataFrame({
    "원래 값": weights.tolist(),
    "정수": q.tolist(),
    "복원값": restored.tolist(),
    "오차": (weights - restored).tolist(),
})
display(int8_example)

0.10은 0.004016으로 나누면 24.9라 정수 25로 저장되고, 복원하면 0.1004가 됩니다(그림과 같은 값). 반올림은 가장 가까운 정수로 옮기므로 오차는 커도 스케일의 절반(약 0.002)을 넘지 않습니다.

### 3-2. 묶음 스케일과 `quantize_symmetric`

스케일을 텐서 전체에 하나만 두면, 절댓값이 아주 큰 값 하나가 모든 값의 눈금 간격을 넓힙니다. 그래서 실제 양자화 형식은 스케일을 작은 단위마다 따로 둡니다.

- **행마다 스케일:** 가중치 행렬의 행(출력 채널. 출력 값 하나를 만드는 가중치 묶음)마다 스케일을 하나씩 둡니다.
- **블록 양자화(block-wise quantization):** 가중치를 32개처럼 작은 묶음(블록)으로 나눠 묶음마다 스케일을 둡니다. 큰 값은 자기 묶음의 눈금만 넓힙니다. 이 교안은 블록을 '묶음'이라고 부릅니다.

<img src="images/교안/34_block_scale.png" width="1050" alt="제목은 '스케일 단위: 텐서 하나와 32개 묶음'입니다. 왼쪽 판 텐서 하나에 스케일 하나: 이상치 하나 때문에 작은 값들이 몇 개 눈금에 몰림. 오른쪽 판 32개 묶음마다 스케일 하나: 이상치는 자기 묶음에만 영향. 결론: GGUF · AWQ · GPTQ는 묶음마다 스케일을 둡니다.">

`quantize_symmetric(weights, bits, group_size)`는 3-1의 계산을 묶음마다 하는 함수입니다. 4·5절에서 실제 층과 모델 전체에 다시 씁니다.

- **`max_int = 2 ** (bits - 1) - 1`:** INT8이면 127, INT4이면 7입니다.
- **`reshape(-1, group_size)`:** 가중치를 앞에서부터 group_size개씩 잘라 한 줄씩 놓습니다. 한 줄이 스케일을 함께 쓰는 한 묶음입니다. 가중치 개수가 group_size로 나누어떨어져야 합니다. group_size가 0이면 텐서 전체를 한 줄로 둡니다.
- **`amax(dim=1, keepdim=True)`:** 줄마다 최대 절댓값을 구해 (줄 수, 1) 모양으로 둡니다. 그래서 나눗셈이 줄마다 자기 스케일로 계산됩니다.

3-1의 가중치 6개로 확인합니다. 스케일 하나(group_size 0)는 3-1의 복원값과 같고, 3개 묶음(group_size 3)은 묶음마다 스케일을 따로 둡니다.

In [ ]:
def quantize_symmetric(weights, bits, group_size):
    """weights를 bits비트로 대칭 양자화했다가 복원한 값을 원래 모양으로 돌려줍니다. group_size가 0이면 스케일을 하나만 둡니다."""
    # bits비트 정수로 담을 가장 큰 값: INT8은 127, INT4는 7
    max_int = 2 ** (bits - 1) - 1

    # 스케일을 함께 쓸 묶음으로 나눕니다. 한 줄이 한 묶음입니다
    if group_size == 0:
        groups = weights.reshape(1, -1)
    else:
        groups = weights.reshape(-1, group_size)

    # 묶음마다 최대 절댓값 / max_int 로 스케일을 정하고, 나누고 반올림해 정수로 만듭니다
    group_scale = groups.abs().amax(dim=1, keepdim=True) / max_int
    group_q = torch.round(groups / group_scale)

    # 정수에 스케일을 다시 곱해 복원하고 원래 모양으로 되돌립니다
    restored_groups = group_q * group_scale
    return restored_groups.reshape(weights.shape)


# 3-1의 6개를 스케일 하나와 3개 묶음으로 INT8 양자화해 복원값과 오차를 비교합니다
group_example = pd.DataFrame({
    "원래 값": weights.tolist(),
    "스케일 하나": quantize_symmetric(weights, 8, 0).tolist(),
    "3개 묶음": quantize_symmetric(weights, 8, 3).tolist(),
})
group_example["스케일 하나 오차"] = group_example["원래 값"] - group_example["스케일 하나"]
group_example["3개 묶음 오차"] = group_example["원래 값"] - group_example["3개 묶음"]
display(group_example)

- **스케일 하나:** 3-1의 복원값과 같습니다.
- **3개 묶음:** 앞 묶음(0.10, -0.42, 0.03)은 최대 절댓값이 0.42라 눈금 간격이 3-1보다 좁고, 뒤 묶음은 0.51이 최대라 3-1과 같습니다. 묶음마다 눈금을 자기 값에 맞추므로 오차가 대체로 작아집니다.
- **저장 비용:** 묶음마다 스케일도 저장해야 합니다. INT8 정수 32개(256비트)에 16비트 스케일 하나를 더하면 가중치 하나당 (256 + 16) ÷ 32 = 8.5비트입니다. 6절에서 볼 GGUF의 `Q8_0`이 이 구조입니다.

## 4. 이상치와 스케일 단위

실제 모델의 가중치에는 다른 값보다 훨씬 큰 소수의 값, **이상치**(outlier)가 섞여 있습니다. 텐서 하나에 스케일 하나를 두면 스케일이 이상치에 맞춰집니다. 6번째 층 FFN의 마지막 선형층 `down_proj`로 확인합니다.

### 4-1. 이상치가 눈금을 넓히는 정도

대부분의 가중치가 정수 눈금을 몇 칸이나 쓰는지 계산합니다.

- **99% 분위:** 작은 값부터 줄 세웠을 때 99% 자리에 있는 값입니다. 100개 중 99개가 이 값보다 작습니다.
- **`kthvalue(k)`:** k번째로 작은 값을 찾습니다. down_proj 가중치는 2560 × 9216 = 약 2,359만 개인데, `torch.quantile`은 원소가 약 1,677만 개를 넘는 텐서를 받지 않아 이 함수로 99% 분위를 구합니다.

In [ ]:
# 6번째 층(인덱스 5) FFN의 마지막 선형층(down_proj) 가중치를 FP32로 꺼냅니다. detach()는 기울기 기록을 떼어 냅니다
down_proj_weights = model.model.layers[5].mlp.down_proj.weight.detach().float()

# 절댓값의 99% 분위(전체 개수의 99% 번째로 작은 값)와 최대 절댓값을 구합니다
abs_weights = down_proj_weights.abs().flatten()
p99_index = int(abs_weights.numel() * 0.99)
p99 = abs_weights.kthvalue(p99_index).values.item()
max_abs = abs_weights.max().item()

# 텐서 전체에 스케일 하나를 둘 때, 가중치의 99%가 쓰는 정수 범위를 봅니다
scale = max_abs / 127
down_proj_stats = {
    "99% 분위 절댓값": p99,
    "최대 절댓값": max_abs,
    "최대 / 99% 분위": max_abs / p99,
    "스케일": scale,
    "99%의 값이 쓰는 정수 범위(±)": p99 / scale,
}
display(pd.Series(down_proj_stats, dtype=object))

**테이블 읽기**
- **이상치의 크기:** `최대 / 99% 분위`가 최대 절댓값이 99% 분위의 몇 배인지입니다.
- **눈금 낭비:** 가중치의 99%는 `99%의 값이 쓰는 정수 범위(±)` 안쪽의 정수만 씁니다. 바깥쪽 칸은 소수의 이상치가 차지하므로, 이상치 몇 개 때문에 나머지 가중치 전체의 오차가 커집니다.

### 4-2. 행마다·묶음마다 스케일

같은 층을 INT8로 텐서 하나, 행마다, 32개 묶음마다 스케일을 두어 양자화합니다. 행마다는 `group_size`에 한 행의 길이(`down_proj_weights.shape[1]`)를 주면 한 줄이 한 행이 됩니다.

In [ ]:
# 같은 층을 세 스케일 단위로 INT8 양자화하고 평균 절대 오차를 비교합니다
row_length = down_proj_weights.shape[1]
int8_rows = []
for method, group_size in [("텐서 하나", 0), ("행마다", row_length), ("32개 묶음", 32)]:
    restored = quantize_symmetric(down_proj_weights, 8, group_size)
    int8_rows.append({"스케일 단위": method, "group_size": group_size, "평균 절대 오차": (down_proj_weights - restored).abs().mean().item()})
display(pd.DataFrame(int8_rows))

스케일 단위가 작을수록 이상치의 영향이 좁아져 평균 절대 오차가 줄어듭니다. 텐서 하나에 비해 몇 분의 1로 줄었는지 출력에서 확인합니다.

## 5. 4비트 오차가 답에 미치는 영향

INT4는 메모리가 INT8의 절반입니다. -8~7의 16개를 담지만 대칭 양자화는 -7~7의 15개만 쓰므로, 스케일을 어디에 두느냐가 더 중요해집니다.

### 5-1. 4비트에서 0이 되는 가중치

텐서 하나 스케일의 INT4는 눈금 간격이 최대 절댓값 ÷ 7이라, 절댓값이 그 절반보다 작은 가중치가 모두 0이 됩니다. 0이 된 가중치의 비율을 셉니다.

In [ ]:
# 같은 층을 INT4로 양자화해 텐서 하나 스케일과 32개 묶음 스케일의 오차와 0이 된 비율을 비교합니다
int4_rows = []
for method, group_size in [("텐서 하나", 0), ("32개 묶음", 32)]:
    restored = quantize_symmetric(down_proj_weights, 4, group_size)
    zero_ratio = (restored == 0).float().mean().item()
    int4_rows.append({"스케일 단위": method, "평균 절대 오차": (down_proj_weights - restored).abs().mean().item(), "0이 된 비율": zero_ratio})
display(pd.DataFrame(int4_rows))

텐서 하나 스케일에서는 4-1의 99% 분위가 눈금 간격의 절반보다 작으면 가중치 대부분이 0이 됩니다. 32개 묶음마다 스케일을 두면 0이 되는 비율과 오차가 얼마나 줄어드는지 출력에서 확인합니다.

### 5-2. 모의 양자화 모델로 답 비교

이제 모델 전체에 적용해 답이 어떻게 달라지는지 봅니다.

- **모의 양자화(fake quantization):** 가중치를 양자화했다가 복원한 값으로 실수 가중치를 바꿔 끼우는 것입니다. 정수로 저장하지 않고 양자화 오차만 모델에 넣어 봅니다.
- **방법:** 모델 사본을 새로 불러, 층(`model.model.layers`) 안의 모든 선형층(`torch.nn.Linear`) 가중치를 양자화했다가 복원한 값으로 바꿉니다. `modules()`는 층 안에 들어 있는 모든 하위 모듈을 차례로 돌려주므로, 어텐션 종류가 달라도 선형층을 모두 찾습니다. 실제 형식도 임베딩은 높은 비트로 남기는 경우가 많아 층 안의 선형층만 바꿉니다.
- **메모리는 그대로:** 복원한 값을 다시 BF16 실수로 저장하므로 메모리는 줄지 않습니다. 답이 어떻게 달라지는지만 보는 시연입니다. 실제로 메모리를 줄이려면 정수로 저장하고 계산하는 전용 형식(6절)이 필요합니다.
- **메모리 비우기:** 원본 답을 만든 뒤 원본 모델을 지우고, 사본을 하나씩 불러 답을 만들고 지웁니다.

먼저 1-3과 같은 헬프데스크 질문으로 원본 모델의 답을 만듭니다. 생성 순서는 교안 01의 Transformers 생성과 같습니다.

In [ ]:
# 추론 모드를 끈 chat template으로 프롬프트 문자열을 만들고 토큰 번호로 바꿉니다. 뒤의 모의 양자화 모델도 이 입력을 그대로 씁니다
prompt = tokenizer.apply_chat_template(helpdesk_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 원본 모델로 greedy 생성하고, 입력 길이 뒤의 새 토큰만 글자로 바꿉니다
output_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False)
new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
original_answer = tokenizer.decode(new_ids, skip_special_tokens=True)
print(original_answer)

In [ ]:
# 원본 모델을 지워 GPU 메모리를 비웁니다. 이 교안의 남은 실습은 원본 모델을 다시 쓰지 않습니다
del model
torch.cuda.empty_cache()

# 원본 답을 첫 행에 두고, INT4 모의 양자화 모델 두 개의 답을 이어 붙입니다
fake_rows = [{"모델": "원본(BF16)", "답": original_answer}]
for method, group_size in [("INT4 텐서 하나", 0), ("INT4 32개 묶음", 32)]:
    # 모델 사본을 새로 불러옵니다. 맨 위 셀에서 내려받은 파일을 캐시에서 읽습니다
    fake_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)

    # 층 안의 선형층마다 가중치를 INT4로 양자화했다가 복원한 값(BF16)으로 덮어씁니다
    with torch.no_grad():
        for module in fake_model.model.layers.modules():
            if isinstance(module, torch.nn.Linear):
                restored = quantize_symmetric(module.weight.float(), 4, group_size)
                module.weight.copy_(restored.to(torch.bfloat16))

    # 원본과 같은 입력으로 greedy 답을 만들고, 다 쓴 사본은 지워 메모리를 돌려줍니다
    output_ids = fake_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    fake_rows.append({"모델": method, "답": tokenizer.decode(new_ids, skip_special_tokens=True)})
    del fake_model
    torch.cuda.empty_cache()

display(pd.DataFrame(fake_rows))

**결과 읽기**
- **INT4 텐서 하나:** 5-1에서 0이 된 비율이 높았다면 답이 원본과 크게 달라지거나 깨집니다. 출력의 답을 원본과 나란히 읽습니다.
- **INT4 32개 묶음:** 원본 답과 얼마나 가까운지 비교합니다. 작은 모델일수록 4비트 오차에 민감한 편이라, 묶음 스케일을 써도 원본과 달라진 곳이 있는지 직접 읽습니다.
- **실제 4비트 형식:** 반올림만 하는 이 모의 양자화보다 오차를 더 줄이는 장치를 씁니다. GGUF `Q4_K_M`은 일부 텐서를 더 높은 비트로 남기고, GPTQ는 양자화로 생긴 오차를 보정하고, AWQ는 중요한 채널을 보호합니다(6절).

## 6. 양자화 형식: AWQ·GPTQ·GGUF

5절의 모의 양자화는 메모리를 줄이지 못했습니다. 실무에서는 정수로 저장하고 오차를 줄이는 장치를 더한 형식을 씁니다. 1-3의 Ollama 태그도 그중 하나(GGUF)입니다.

### 6-1. AWQ·GPTQ

- **학습 후 양자화(post-training quantization, PTQ):** 학습을 마친 모델을 다시 학습하지 않고 가중치만 양자화하는 것입니다.
- **보정 데이터(calibration data):** 양자화할 때 모델에 넣어 볼 예시 문장 묶음입니다. 실제 입력이 들어왔을 때 어느 가중치가 답에 크게 영향을 주는지 보는 데 씁니다.
- **AWQ(Activation-aware Weight Quantization):** 활성값(층에 들어오는 입력 값)을 보고 답에 중요한 가중치를 찾아 보호하는 학습 후 양자화 방법입니다. 중요한 가중치를 미리 키워 둔 뒤 양자화해 그 가중치의 오차를 줄입니다.
- **GPTQ:** 학습 후 양자화 방법입니다. 가중치를 한 열씩 양자화하면서, 그때 생긴 오차를 아직 양자화하지 않은 가중치를 고쳐 보정합니다.
- **공통:** 주로 4비트로 줄이고, 3-2처럼 묶음마다 스케일을 둡니다. 결과는 safetensors 파일로 저장해 GPU 서버에서 전용 실행 도구로 씁니다.

<img src="images/교안/35_awq_gptq.png" width="1050" alt="제목은 'AWQ와 GPTQ: 학습을 마친 모델을 양자화하는 두 방법'입니다. 공통: 보정 데이터를 넣어 보고, 주로 4비트로 줄입니다. AWQ(중요한 채널을 보호합니다): 1 활성값이 큰 채널 찾기, 2 그 채널의 가중치를 미리 키우기, 3 모두 같은 비트로 양자화. GPTQ(생긴 오차를 남은 가중치로 보정합니다): 1 한 열씩 양자화, 2 생긴 오차 계산, 3 남은 열을 고쳐 오차 줄이기. 두 판 아래: 만들 때 GPU와 전용 도구가 필요합니다.">

### 6-2. Hub에서 양자화 저장소 찾기

양자화한 저장소는 원본 모델 페이지 오른쪽의 Model tree에서 찾습니다. `Quantizations`를 누르면 이 모델을 AWQ·GPTQ·GGUF 등으로 바꿔 올린 저장소 목록이 나옵니다.

<img src="images/교안/cap_hf_06_model_tree.png" width="1050" alt="Hugging Face의 Qwen/Qwen3.5-4B 모델 페이지 오른쪽 Model tree 화면입니다. Base model은 Qwen/Qwen3.5-4B-Base이고, 아래로 Adapters, Finetunes, Merges, Quantizations 줄이 이어집니다. 빨간 사각형은 이 모델을 양자화해 올린 저장소 수를 보여 주는 Quantizations 줄을 표시합니다.">

- 누를 곳: 모델 페이지 -> Model card 탭 오른쪽의 Model tree
- 주소: https://huggingface.co/Qwen/Qwen3.5-4B

`Quantizations`를 누르면 양자화 저장소 목록이 열립니다. 정렬을 내려받기 순(Most downloads)으로 바꾸면 많이 쓰이는 저장소가 위에 옵니다.

<img src="images/교안/cap_hf_06b_quantized_list.png" width="1050" alt="Quantizations를 눌러 연 양자화 저장소 목록 화면입니다. 왼쪽 빨간 사각형은 Model Tree의 Qwen/Qwen3.5-4B와 선택된 Quantizations 필터, 오른쪽 빨간 사각형은 내려받기 순으로 정렬한 목록의 위 두 저장소 unsloth/Qwen3.5-4B-GGUF와 sanskar003/Qwen3.5-4B-AWQ를 표시합니다.">

- 누를 곳: Model tree -> Quantizations -> 오른쪽 위 Sort를 Most downloads로
- 주소: https://huggingface.co/models?other=base_model:quantized:Qwen/Qwen3.5-4B&sort=downloads

- **공식 저장소가 없음:** Qwen은 Qwen3.5-4B의 양자화 저장소(AWQ·GPTQ·GGUF·FP8)를 직접 올리지 않았습니다. 목록은 모두 사용자(커뮤니티)가 올린 저장소입니다.
- **커뮤니티 예:** `unsloth/Qwen3.5-4B-GGUF`(GGUF 파일 묶음), `sanskar003/Qwen3.5-4B-AWQ`·`cyankiwi/Qwen3.5-4B-AWQ-4bit`(AWQ), `RedHatAI/Qwen3.5-4B-FP8-dynamic`(8비트 부동소수 FP8)
- **쓰기 전 확인:** 개발사가 아닌 저장소는 원본(`base_model`)과 라이선스를 모델 카드에서 확인합니다. 이런 저장소를 실행하려면 형식마다 전용 실행 도구가 필요하므로, 이 교안은 찾는 방법까지만 다룹니다.

### 6-3. GGUF 양자화 태그

**GGUF**는 교안 01 1-1에서 정의한 형식입니다. llama.cpp와 Ollama가 쓰는 파일 형식으로, 양자화한 가중치와 토크나이저·설정을 파일 하나에 담습니다. 파일 이름과 Ollama 태그에 붙는 이름이 양자화 방식을 나타냅니다.

| 태그 | 뜻 | 가중치당 비트(대략) |
|---|---|---|
| `BF16` | 양자화하지 않은 16비트 | 16 |
| `Q8_0` | 32개 묶음마다 스케일 하나를 둔 8비트(3-2와 같은 구조) | 8.5 |
| `Q4_0` | 32개 묶음마다 스케일 하나만 둔 단순 4비트(옛 형식, 보통은 `Q4_K_M` 권장) | 4.5 |
| `Q4_K_M` | k-quant 방식 4비트. 묶음을 두 단계로 나눠 스케일을 담고, 일부 텐서는 6비트 이상으로 남긴 중간 크기(M) | 약 4.9 |

Hub의 GGUF 저장소에서 형식마다 파일 하나씩 올라온 것을 볼 수 있습니다. 6-2 목록의 첫 저장소 `unsloth/Qwen3.5-4B-GGUF`를 엽니다.

<img src="images/교안/cap_hf_07_gguf_files.png" width="1050" alt="Hugging Face의 unsloth/Qwen3.5-4B-GGUF 저장소 Files and versions 화면입니다. Qwen3.5-4B-BF16.gguf부터 Q3_K_S, Q4_0, Q4_K_M, Q5_K_M, Q6_K, Q8_0까지 형식마다 GGUF 파일 하나씩 있습니다. 빨간 사각형은 Qwen3.5-4B-Q4_K_M.gguf(2.74 GB)와 Qwen3.5-4B-Q8_0.gguf(4.48 GB) 줄을 표시합니다.">

- 누를 곳: 6-2 목록 -> unsloth/Qwen3.5-4B-GGUF -> Files and versions 탭
- 주소: https://huggingface.co/unsloth/Qwen3.5-4B-GGUF/tree/main

- **실습 모델의 태그:** Ollama 라이브러리의 Qwen3.5-4B 모델 파일은 `q8_0`이 4,482,403,136바이트, `q4_K_M`이 2,648,593,216바이트입니다. 글 생성 부분 파라미터 4,205,751,296개로 나누면 가중치당 약 8.5비트와 약 5.0비트입니다. `Q4_K_M`이 4비트보다 큰 것은 묶음 스케일과 더 높은 비트로 남긴 텐서 때문입니다.
- **같은 이름, 다른 파일:** Hub 화면의 `Q8_0`(4.48 GB)은 Ollama 모델 파일과 거의 같지만, `Q4_K_M`(2.74 GB)은 Ollama 파일(약 2.65 GB)보다 조금 큽니다. 같은 `Q4_K_M`이라도 만든 곳마다 더 높은 비트로 남기는 텐서가 달라 크기가 조금씩 다릅니다.

Ollama 라이브러리의 태그 목록에서도 같은 모델의 양자화 수준별 태그와 크기를 볼 수 있습니다.

<img src="images/교안/cap_ollama_02_tags.png" width="1050" alt="Ollama 라이브러리의 qwen3.5 태그 목록 화면입니다. 빨간 사각형은 qwen3.5:4b-q4_K_M(3.3GB), qwen3.5:4b-q8_0(5.2GB), qwen3.5:4b-bf16(9.3GB) 줄을 표시하고, 세 줄 모두 컨텍스트 256K, 입력 Text, Image입니다.">

- 누를 곳: qwen3.5 페이지 -> Models 테이블 오른쪽 위 View all
- 주소: https://ollama.com/library/qwen3.5/tags

- **화면의 크기:** qwen3.5 태그는 사진을 읽는 부분(비전 projector, 675,569,120바이트)도 함께 받으므로, 화면의 크기는 위 모델 파일 크기에 이 값을 더한 것입니다.

### 6-4. 형식 선택

어디서 실행하는지에 따라 형식이 정해지고, 정확도는 같은 질문 묶음으로 직접 비교해 확인합니다.

<img src="images/교안/36_quant_choice.png" width="1050" alt="제목은 '양자화 형식 고르기: 실행 환경에 따라'입니다. BF16 원본: 파일 safetensors, 가중치당 16비트, 어디서 GPU · 파인튜닝, 정확도 기준. AWQ · GPTQ: 파일 safetensors, 가중치당 4비트 + 스케일, 어디서 GPU 서버, 정확도 원본에 가까움. GGUF: 파일 .gguf 하나, 가중치당 8.5비트 · 4.9비트, 어디서 개인 PC · Ollama, 정확도 태그마다 다름. 개발자가 세 카드를 가리킴. 결론: 작을수록 가볍고 빠르지만, 정확도는 같은 질문으로 비교합니다.">

- **파인튜닝:** 튜닝은 BF16 원본으로 하고, 튜닝한 모델을 AWQ·GPTQ로 양자화하거나 GGUF로 변환해 배포합니다.

## 7. 태그 비교 결과 읽기

1-3의 `measure_tag`가 두 태그를 어떻게 비교했는지 보고, 결과를 읽습니다. 토큰/초는 교안 01 4-1과 같은 계산(`eval_count` / `eval_duration`)입니다. 응답에는 모델을 불러오는 시간 `load_duration`(나노초)이 따로 담깁니다.

- **첫 호출 버리기:** 첫 호출은 모델을 GPU에 올리고 준비하는 일이 겹쳐 생성 속도가 흔들립니다. 불러오는 시간 자체는 `load_duration`에 따로 담기지만, 생성 속도도 흔들리므로 결과를 버립니다.
- **중앙값:** 한 번 측정한 값은 흔들리므로 3번 측정해 중앙값(가운데 값)을 씁니다.
- **측정 뒤 내리기:** 측정을 마친 태그는 `keep_alive=0`으로 GPU에서 내려, 다음 태그를 같은 조건에서 측정합니다.
- **같은 답 설정:** temperature 0이면 매번 가장 확률이 높은 토큰을 고르고, seed는 난수를 고정합니다. 추론 모드는 교안 01처럼 `think=False`로 끕니다.

**1-3 결과 읽기**
- **속도:** 토큰 하나를 만들 때마다 가중치 전체를 GPU 메모리에서 읽어야 해서, 읽을 바이트가 적은 4비트 태그가 빠릅니다. 읽을 바이트가 줄어든 만큼(파일 크기 비율에 가깝게) 빨라지는지 1-3의 테이블과 6-3의 모델 파일 크기로 비교합니다(비전 projector는 글을 생성할 때 읽지 않습니다). 토큰/초 값은 GPU 종류와 그때 장치가 하는 다른 일에 따라 달라집니다.
- **답:** 같은 설정(temperature 0, seed 0)인데도 태그마다 문장이 다를 수 있습니다. 가중치 값이 달라졌기 때문입니다. 비트를 줄일수록 원본 답과 달라지기 쉬우므로, 고를 때는 속도와 함께 우리 질문 묶음의 답을 비교합니다.

## 8. 🖐️ 함께 따라하기: 법무팀 계약서 검토 모델의 장비 판정

본실습의 메모리 계산과 태그 비교를 법무팀 계약서 검토 모델에 적용해, 사내 GPU 한 장에 올릴 크기와 형식을 판정합니다.

**본실습과 다른 점**
- **장비와 후보:** 사내 GPU는 24GB 한 장이고, 후보는 1.5B·7B·14B·32B 모델입니다. 계약서는 밖으로 보낼 수 없어 사내 GPU에서 실행합니다.
- **질문:** 계약서 조항 검토 질문 하나를 `q8_0`과 `q4_K_M` 두 태그에 넣습니다.
- **태그 비교:** 후보 모델의 Ollama 태그 대신, 받아 둔 실습 모델의 두 태그로 같은 계약서 질문에서 비트에 따른 속도와 답의 차이를 확인합니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_02_정답.ipynb`의 같은 절에 있습니다.

**설정**
- **GPU 판정:** 2-4처럼 GPU 사양 24GB를 24 GiB로 읽고, 가중치 메모리에 20%를 더한 생성 시 메모리가 24 이하면 "들어감", 아니면 "안 들어감"입니다.

### 8-1. 메모리 테이블과 GPU 판정

In [ ]:
legal_gpu_gib = 24

legal_rows = []
for size_name, n_params in [("1.5B", 1.5e9), ("7B", 7e9), ("14B", 14e9), ("32B", 32e9)]:
    for precision in ["BF16", "INT8", "INT4"]:
        # 🖐️ 함께 따라하기: model_memory_gib와 precision_bytes로 가중치 메모리를 계산하고, 20%를 더한 생성 시 메모리를 계산하세요
        legal_weight_gib = ...
        legal_total_gib = ...

        # 🖐️ 함께 따라하기: legal_total_gib가 legal_gpu_gib 이하이면 "들어감", 아니면 "안 들어감"을 legal_fits에 담으세요 (if문)
        ...

        legal_rows.append({"모델 크기": size_name, "정밀도": precision, "가중치(GiB)": legal_weight_gib, "생성 시(GiB)": legal_total_gib, "24GB GPU": legal_fits})

legal_memory = pd.DataFrame(legal_rows)
display(legal_memory.round(2))

### 8-2. 태그 속도와 답 비교

In [ ]:
legal_messages = [
    {"role": "system", "content": "당신은 법무팀의 계약서 검토를 돕는 안내원입니다. 한국어로 짧고 정확하게 답하세요."},
    {"role": "user", "content": "계약 해지 통보 기간 조항을 검토할 때 확인할 점을 두 가지 알려 주세요."},
]

legal_speed_rows = []
for tag in ["qwen3.5:4b-q8_0", "qwen3.5:4b-q4_K_M"]:
    # 🖐️ 함께 따라하기: measure_tag에 tag와 legal_messages를 넣어 토큰/초 중앙값과 답을 받으세요
    legal_tokens_per_sec, legal_answer = ...

    legal_speed_rows.append({"태그": tag, "토큰/초 중앙값": legal_tokens_per_sec, "답": legal_answer})

legal_speed = pd.DataFrame(legal_speed_rows)
display(legal_speed.round(1))

### 8-3. 판단 쓰기

**판단** (두 테이블을 보고 어떤 크기와 형식을 고를지 두세 문장으로 쓰세요)

- ...

## 9. 핵심 정리

- **메모리 어림:** 가중치 메모리 = 파라미터 수 × 파라미터 하나의 바이트(FP32 4, BF16 2, INT8 1, INT4 0.5)이고, 생성할 때는 약 20%를 더합니다. Qwen3.5-4B의 글 생성 부분은 BF16 약 7.83GiB입니다.
- **BF16과 FP16:** 같은 16비트지만 BF16은 FP32와 범위가 같고 자릿수가 적으며, FP16은 자릿수가 조금 더 많지만 최댓값이 65504입니다. 최근 LLM은 BF16으로 배포합니다.
- **대칭 양자화와 스케일:** `scale = 최대 절댓값 / 127`(4비트는 7)로 나누고 반올림해 정수로 저장하고, 스케일을 곱해 복원합니다. 복원값과 원래 값의 차이가 양자화 오차입니다.
- **묶음 스케일과 이상치:** 소수의 이상치가 텐서 전체의 스케일을 키워 작은 값들의 오차를 늘립니다. 32개 묶음마다 스케일을 두면 오차가 크게 줄어듭니다. 4비트에서 텐서 하나에 스케일 하나를 두면 이상치에 맞춘 눈금 간격의 절반보다 작은 가중치가 모두 0이 되고, 묶음 스케일은 이 비율과 오차를 줄입니다.
- **AWQ·GPTQ·GGUF:** AWQ·GPTQ는 보정 데이터로 양자화 오차를 줄이는 학습 후 양자화이고 GPU 서빙에 씁니다. GGUF는 Ollama가 쓰는 파일 하나짜리 형식이며 `Q8_0`·`Q4_K_M` 같은 태그로 양자화 방식을 나타냅니다. 양자화 저장소는 Hub 모델 페이지의 Model tree에서 찾고, 공식 저장소가 없으면 커뮤니티 저장소의 원본과 라이선스를 확인합니다.
- **태그 비교:** 크기, 토큰/초(`eval_count` ÷ `eval_duration`), 같은 질문의 답을 함께 비교합니다. 작은 태그가 빠르지만 답이 달라질 수 있습니다.
- **형식 선택:** GPU 서버 서빙은 AWQ·GPTQ, 개인 PC와 Ollama는 GGUF를 쓰고, 정확도는 BF16 원본을 기준으로 같은 질문 묶음에서 직접 비교합니다.

다음 단원에서는 헬프데스크 학습 데이터셋을 만들고, 튜닝할 모델의 chat template으로 학습 형식을 맞춥니다.